# Cross-View Geolocation — Training Notebook

Train satellite and phone encoders for cross-view geolocation using
the CV-Cities dataset and symmetric InfoNCE loss.

**Runtime:** Google Colab with T4 GPU (free tier)

**Pipeline:**
1. Clone repo & install dependencies
2. Download CV-Cities dataset (seattle, london, tokyo, sydney)
3. Train ResNet-50 teacher encoders (~3-5 hrs)
4. Distill to MobileNetV3 student (~1 hr)
5. Export to ONNX + TFLite (int8 quantized)
6. Download trained models

**Note:** Checkpoints are saved to Google Drive so training survives Colab disconnects. Re-run this notebook to resume.

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repo
!git clone https://github.com/farmino1/cross-view-geolocator.git
%cd cross-view-geolocator

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q huggingface_hub onnx onnxruntime onnx-tf

In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/geolocator'
!mkdir -p $WORK_DIR/checkpoints/teacher
!mkdir -p $WORK_DIR/checkpoints/student
!mkdir -p $WORK_DIR/exported

In [ ]:
# Login to HuggingFace (needed for CV-Cities dataset)
# Get your token from https://huggingface.co/settings/tokens
import os
os.environ['HF_TOKEN'] = ''  # <-- paste your token here

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

## 2. Download CV-Cities Dataset

Downloads ~10GB total. Each zip extracts to `data/<city>/sat_images/` and `data/<city>/pano_images/`.

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

CITIES = ['seattle', 'london', 'tokyo', 'sydney']
DATA_DIR = '/content/data'

for city in CITIES:
    # Check if already downloaded (supports resume)
    sat_dir = f'{DATA_DIR}/{city}/sat_images'
    alt_dir = f'{DATA_DIR}/{city}/satellite'
    if os.path.isdir(sat_dir) or os.path.isdir(alt_dir):
        print(f'{city}: already downloaded, skipping')
        continue
    print(f'Downloading {city}...')
    zip_path = hf_hub_download(
        repo_id='gaoshuang98/CV-Cities',
        filename=f'{city}.zip',
        repo_type='dataset',
    )
    print(f'  Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print(f'  Done.')

print('\nDataset ready.')

In [ ]:
# Verify directory structure
# CV-Cities zips extract to sat_images/ and pano_images/
import os
for city in CITIES:
    city_dir = f'{DATA_DIR}/{city}'
    for name in ['sat_images', 'satellite']:
        p = f'{city_dir}/{name}'
        if os.path.isdir(p):
            sat = len(os.listdir(p))
            break
    else:
        sat = 0
    for name in ['pano_images', 'streetview']:
        p = f'{city_dir}/{name}'
        if os.path.isdir(p):
            street = len(os.listdir(p))
            break
    else:
        street = 0
    print(f'{city}: {sat} satellite, {street} streetview')

## 3. Train Teacher Encoders (ResNet-50)

Both satellite and phone encoders are ResNet-50 projecting to shared 256-dim space.
Loss: Symmetric InfoNCE with learnable temperature (init 0.07).

Checkpoints save to Google Drive every epoch. If Colab disconnects, re-run this cell to resume.

Estimated time: ~3-5 hours on T4 (20 epochs, batch 128).

In [ ]:
from src.training.train import train

resume_ckpt = f'{WORK_DIR}/checkpoints/teacher/latest_checkpoint.pt'

train(
    data_dir=DATA_DIR,
    cities=CITIES,
    output_dir=f'{WORK_DIR}/checkpoints/teacher',
    epochs=20,
    batch_size=128,
    lr=3e-4,
    weight_decay=0.2,
    embed_dim=256,
    warmup_epochs=1,
    checkpoint_interval=5,
    resume_from=resume_ckpt if os.path.exists(resume_ckpt) else None,
    device='cuda',
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt
import json

with open(f'{WORK_DIR}/checkpoints/teacher/history.json') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
r1 = [h.get('recall@1', 0) for h in history]
r5 = [h.get('recall@5', 0) for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')

ax2.plot(epochs, r1, label='R@1')
ax2.plot(epochs, r5, label='R@5')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Recall')
ax2.set_title('Validation Recall@K')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Distill to MobileNetV3

Student (MobileNetV3-Large) learns to mimic the teacher's phone encoder embeddings.

Estimated time: ~1 hour on T4 (10 epochs, batch 128).

In [ ]:
from src.training.distill import distill

distill(
    teacher_checkpoint=f'{WORK_DIR}/checkpoints/teacher/best_model.pt',
    data_dir=DATA_DIR,
    cities=CITIES,
    output_dir=f'{WORK_DIR}/checkpoints/student',
    epochs=10,
    batch_size=128,
    lr=1e-4,
    embed_dim=256,
    device='cuda',
)

## 5. Export to ONNX + TFLite

Exports:
- `satellite_encoder.onnx` — ResNet-50 for desktop tile encoding
- `phone_encoder_teacher.onnx` — ResNet-50 teacher reference
- `phone_encoder.onnx` — MobileNetV3 student for mobile
- `phone_encoder.tflite` — int8 quantized MobileNetV3 for Android

In [ ]:
from src.training.export import export

export(
    checkpoint_path=f'{WORK_DIR}/checkpoints/teacher/best_model.pt',
    output_dir=f'{WORK_DIR}/exported',
    embed_dim=256,
    export_satellite=True,
    export_phone=True,
    quantize_tflite=True,
)

## 6. Quick Test

Verify the trained encoders produce aligned embeddings.

In [ ]:
import torch
import numpy as np
from src.training.models import create_resnet50_encoder, create_mobilenetv3_encoder

teacher_ckpt = torch.load(f'{WORK_DIR}/checkpoints/teacher/best_model.pt', map_location='cpu', weights_only=False)

sat = create_resnet50_encoder(embed_dim=256, pretrained=False)
sat.load_state_dict(teacher_ckpt['sat_encoder'])
sat.eval()

phone = create_resnet50_encoder(embed_dim=256, pretrained=False)
phone.load_state_dict(teacher_ckpt['phone_encoder'])
phone.eval()

dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    sat_emb = sat(dummy)
    phone_emb = phone(dummy)
    sim = torch.nn.functional.cosine_similarity(sat_emb, phone_emb)

print(f'Satellite embedding: {sat_emb.shape}, norm={sat_emb[0].norm():.4f}')
print(f'Phone embedding: {phone_emb.shape}, norm={phone_emb[0].norm():.4f}')
print(f'Pairwise similarity: {[f"{s:.4f}" for s in sim.tolist()]}')

# Test student
student_ckpt_path = f'{WORK_DIR}/checkpoints/student/best_student.pt'
if os.path.exists(student_ckpt_path):
    sckpt = torch.load(student_ckpt_path, map_location='cpu', weights_only=False)
    student = create_mobilenetv3_encoder(embed_dim=256, pretrained=False)
    student.load_state_dict(sckpt['student'])
    student.eval()
    with torch.no_grad():
        s_emb = student(dummy)
        sim_st = torch.nn.functional.cosine_similarity(phone_emb, s_emb)
    print(f'\nStudent embedding: {s_emb.shape}, norm={s_emb[0].norm():.4f}')
    print(f'Student-Teacher alignment: {[f"{s:.4f}" for s in sim_st.tolist()]}')

## 7. Download Models

The files you need:
- `phone_encoder.tflite` — goes on the phone
- `satellite_encoder.onnx` — used with `package_area.py` to build tile indices

In [ ]:
!ls -lh $WORK_DIR/exported/

# Copy to Colab root for easy download
!cp $WORK_DIR/exported/phone_encoder.tflite /content/
!cp $WORK_DIR/exported/satellite_encoder.onnx /content/
!cp $WORK_DIR/exported/phone_encoder.onnx /content/

print('\nFiles ready for download:')
print('  phone_encoder.tflite      — MobileNetV3 int8 for Android')
print('  satellite_encoder.onnx    — ResNet-50 for desktop tile encoding')
print('  phone_encoder.onnx        — MobileNetV3 float32 reference')

In [ ]:
# Download files from Colab
from google.colab import files
files.download('/content/phone_encoder.tflite')
files.download('/content/satellite_encoder.onnx')